In [ ]:
 # Gerekli kütüphaneleri içe aktarma
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
import os

# --- 1. Ortamı Kurma ve Model için Klasörler Oluşturma ---
# Tensorboard logları ve eğitilmiş modeller için klasörler oluşturuyoruz.
log_dir = "logs/"
model_dir = "models/DQN/"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# CartPole-v1 ortamını Gymnasium kütüphanesi ile oluşturuyoruz.
# Bu ortamda amaç, bir çubuğu araba üzerinde dengede tutmaktır.
env = gym.make("CartPole-v1")

# --- 2. DQN Modelini Tanımlama ---
# Stable-Baselines3 kütüphanesinden DQN modelini kullanıyoruz.
# MlpPolicy: Standart bir Multi-Layer Perceptron (çok katmanlı algılayıcı) ağı kullanacağımızı belirtir.
# env: Modelin hangi ortamda eğitileceğini belirtir.
# verbose=1: Eğitim sırasında ilerlemeyi konsola yazdırır.
# tensorboard_log: Tensorboard loglarının nereye kaydedileceğini belirtir.
model = DQN(
    "MlpPolicy",
    env,
    verbose=1,
    tensorboard_log=log_dir
)

# --- 3. Modeli Eğitme ---
# Modeli toplam 10,000 adım (timestep) boyunca eğitiyoruz.
# Her eğitim döngüsünde Tensorboard'a log atılacaktır.
# İsimlendirme (tb_log_name), logların Tensorboard'da kolayca bulunmasını sağlar.
model.learn(total_timesteps=10000, tb_log_name="DQN_CartPole")

# --- 4. Eğitilmiş Modeli Kaydetme ---
# Eğitim tamamlandıktan sonra, modeli il
# eride kullanmak üzere kaydediyoruz.
model_path = os.path.join(model_dir, "dqn_cartpole_model")
model.save(model_path)

print(f"Model {model_path} adresine kaydedildi.")

# --- 5. Modeli Değerlendirme ---
# Eğitilmiş modelin performansını 10 bölüm (episode) boyunca değerlendiriyoruz.
# evaluate_policy fonksiyonu, modelin ortalama ödülünü ve standart sapmasını döndürür.
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Ortalama Ödül: {mean_reward:.2f} +/- {std_reward:.2f}")



: 

In [ ]:
# --- 6. Eğitilmiş Modeli Test Etme (Daha Kararlı Görselleştirme) ---
import gymnasium as gym
from stable_baselines3 import DQN
import os
import time

# --- Modelin kaydedildiği yolu belirt ---
model_dir = "models/DQN/"
model_path = os.path.join(model_dir, "dqn_cartpole_model.zip") # .zip uzantısını ekledim, SB3 varsayılan olarak bunu kullanır

# Eğer model yolu mevcut değilse, hata ver
if not os.path.exists(model_path):
    print(f"Hata: Model dosyası '{model_path}' bulunamadı. Lütfen önce modeli eğitin.")
else:
    print("\nEğitilmiş model test ediliyor...")

    # Modeli yeniden yükle
    model = DQN.load(model_path)

    # Görselleştirme için TEK bir standart ortam oluştur
    # Not: 'human' modu bir pencere açar.
    env = gym.make("CartPole-v1", render_mode="human")

    # Kaç bölüm test edileceğini belirt
    episodes_to_run = 5

    for episode in range(episodes_to_run):
        # Ortamı her bölüm başında sıfırla
        obs, info = env.reset()
        
        # Bölüm sonu bayrakları
        terminated = False
        truncated = False
        
        episode_reward = 0
        
        # Bölüm bitene kadar döngüyü sürdür
        while not terminated and not truncated:
            # Ortamı render et (görselleştir)
            env.render()
            
            # Ajanın aksiyonunu tahmin et
            action, _states = model.predict(obs, deterministic=True)
            
            # Ortamda bir adım at
            obs, reward, terminated, truncated, info = env.step(action)
            
            episode_reward += reward
            
            # Görselin daha akıcı olması için küçük bir bekleme
            time.sleep(0.01)

        print(f"Bölüm {episode + 1} tamamlandı. Toplam Ödül: {episode_reward}")

    # Tüm bölümler bittiğinde ortamı düzgünce kapat
    env.close()
    print("Görselleştirme tamamlandı.")